## Currency Conversion Tool

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests, json

In [ ]:
# tool create

In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'exchange-rate-api/{base_currency}/{target_currency}'
    
    response = requests.get(url)
    
    return response.json()
    

In [ ]:
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    
    return base_currency_value * conversion_rate

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95})

# tool bind

In [ ]:
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools({'base_currency_value':10, 'conversion_rate':95})

In [ ]:
query = HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 uSD into INR')

In [ ]:
messages = [query]
messages

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

In [ ]:
for tool_call in ai_message.tool_calls:
    
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message1)
        

In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content